# Notebook 06 – Identifying Epistemic Enclaves

**DigitAfrica Workshop 2026 – Identifying Epistemic Enclaves and Understanding Polarisation**

## Learning objectives
- Define and operationalise *epistemic enclaves* in a social network
- Detect enclaves by combining structural and content-based features
- Characterise enclaves by their information diet and internal cohesion

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

%matplotlib inline

## 1. Build feature matrix for nodes

We combine structural features (degree, clustering coefficient) with content features (opinion score).

In [ ]:
rng = np.random.default_rng(1)
n_nodes = 100
groups = {i: ("A" if i < 50 else "B") for i in range(n_nodes)}

G = nx.Graph()
G.add_nodes_from(groups.keys())
nx.set_node_attributes(G, groups, "group")

for u in range(n_nodes):
    for v in range(u + 1, n_nodes):
        same_group = groups[u] == groups[v]
        prob = 0.08 if same_group else 0.01
        if rng.random() < prob:
            G.add_edge(u, v)

opinion = {
    n: rng.normal(-0.6 if groups[n] == "A" else 0.6, 0.15)
    for n in G.nodes()
}
nx.set_node_attributes(G, opinion, "opinion")

# Build feature matrix
clustering = nx.clustering(G)
node_features = pd.DataFrame({
    "node": list(G.nodes()),
    "degree": [G.degree(n) for n in G.nodes()],
    "clustering": [clustering[n] for n in G.nodes()],
    "opinion": [opinion[n] for n in G.nodes()],
    "group": [groups[n] for n in G.nodes()],
})
node_features.head()

## 2. Cluster nodes into enclaves

In [ ]:
features = ["degree", "clustering", "opinion"]
X = StandardScaler().fit_transform(node_features[features])

kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
node_features["enclave"] = kmeans.fit_predict(X)

print(node_features.groupby("enclave")[features].mean().round(3))

## 3. Characterise the enclaves

In [ ]:
for enc_id, enc_df in node_features.groupby("enclave"):
    print(f"\n--- Enclave {enc_id} ---")
    print(f"  Size            : {len(enc_df)}")
    print(f"  Mean opinion    : {enc_df['opinion'].mean():.3f}")
    print(f"  Mean degree     : {enc_df['degree'].mean():.2f}")
    print(f"  Mean clustering : {enc_df['clustering'].mean():.3f}")
    print(f"  Group breakdown : {enc_df['group'].value_counts().to_dict()}")

## 4. Visualise enclaves on the network

In [ ]:
enc_map = dict(zip(node_features["node"], node_features["enclave"]))
color_map = ["gold" if enc_map[n] == 0 else "mediumpurple" for n in G.nodes()]
pos = nx.spring_layout(G, seed=42)

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx(
    G, pos=pos, node_color=color_map,
    node_size=60, with_labels=False,
    edge_color="grey", alpha=0.7, ax=ax
)
ax.set_title("Epistemic enclaves (yellow = Enclave 0, purple = Enclave 1)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Wrap-up & discussion

- What are the limitations of the K-Means approach for enclave detection?
- How would you extend this analysis to a temporal network?
- What interventions might reduce the isolation of epistemic enclaves?